# PCA and LDA: Synthetic Dairy Milking Dataset

**Purpose:** Learn PCA and Linear Discriminant Analysis (LDA) in a small, reproducible example inspired by the MSU dairy influenza early-detection project.

> **Important:** Every row in this notebook is synthetic. These are invented observations, not MSU farm records, not real biological measurements, and not evidence about influenza. We use them only to learn the workflow.

## Learning goals
1. Explore milking-event measurements from cows.
2. Standardize features because they use different units and scales.
3. Use PCA to find directions of high overall variation.
4. Use LDA to find directions that separate the supplied classes.
5. Compare the two projections and discuss limitations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("Libraries loaded.")

## 1. Create a synthetic dataset

We simulate 80 cows, each with 12 milking events. Each event has four measurements:

- `milk_yield_kg`: milk yield in kilograms
- `milk_temperature_c`: milk temperature in Celsius
- `conductivity`: an invented conductivity-like measurement
- `visits_per_day`: number of voluntary visits per day

Half the cows are assigned a retrospective `cow_status` of `infected`, and half `not_infected`. For the infected group, later events are generated with a stronger shift in some measurements. This gives us a deliberately simplified teaching example.

**Important modeling detail:** the label is assigned at the cow level, while rows represent milking events. This mirrors one challenge in the research description: cow-level retrospective outcomes do not tell us the exact onset time for each event. Our synthetic data has a simplified pattern that real data will not necessarily have.

In [ ]:
n_cows = 80
events_per_cow = 12

records = []

for cow_num in range(n_cows):
    cow_id = f"C{cow_num+1:03d}"
    infected = cow_num < n_cows // 2
    status = "infected" if infected else "not_infected"

    # Cow-specific baseline differences
    baseline_yield = rng.normal(34, 4)
    baseline_temp = rng.normal(38.5, 0.25)
    baseline_cond = rng.normal(5.2, 0.35)
    baseline_visits = rng.normal(4.5, 0.6)

    for event in range(events_per_cow):
        # A gradual synthetic shift in the infected group during later events
        progression = max(0, (event - 5) / 6) if infected else 0

        milk_yield = baseline_yield - 7.0 * progression + rng.normal(0, 1.8)
        milk_temp = baseline_temp + 0.75 * progression + rng.normal(0, 0.18)
        conductivity = baseline_cond + 0.9 * progression + rng.normal(0, 0.25)
        visits = baseline_visits - 1.1 * progression + rng.normal(0, 0.35)

        records.append({
            "cow_id": cow_id,
            "event_number": event + 1,
            "cow_status": status,
            "milk_yield_kg": max(0.5, milk_yield),
            "milk_temperature_c": milk_temp,
            "conductivity": conductivity,
            "visits_per_day": max(0.5, visits)
        })

df = pd.DataFrame(records)

print("Dataset shape:", df.shape)
display(df.head(10))

In [ ]:
print("Rows per class:")
display(df.groupby("cow_status").size())

print("\nSummary statistics:")
display(df[["milk_yield_kg", "milk_temperature_c", "conductivity", "visits_per_day"]].describe().round(2))

## 2. Explore the data

First, compare the feature distributions by the retrospective cow-level label. Remember that these plots show the artificial shifts we built into the simulation.

In [ ]:
features = [
    "milk_yield_kg",
    "milk_temperature_c",
    "conductivity",
    "visits_per_day"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for feature, ax in zip(features, axes.ravel()):
    sns.boxplot(data=df, x="cow_status", y=feature, ax=ax)
    ax.set_title(feature.replace("_", " ").title())
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 3. Prepare the feature matrix

PCA and LDA operate on numerical features, not IDs or text labels. We exclude `cow_id`, `event_number`, and `cow_status` from the feature matrix.

We standardize the features so that measurements with larger numerical scales do not dominate simply because of their units. The scaler is fitted on the training data only to avoid data leakage.

In [ ]:
X = df[features]
y = df["cow_status"]

# Split by cow, not by individual row, so events from one cow
# cannot appear in both train and test sets.
cow_labels = df[["cow_id", "cow_status"]].drop_duplicates()
train_cows, test_cows = train_test_split(
    cow_labels["cow_id"],
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=cow_labels["cow_status"]
)

train_mask = df["cow_id"].isin(train_cows)
test_mask = df["cow_id"].isin(test_cows)

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training events:", len(X_train))
print("Testing events:", len(X_test))
print("Training cows:", len(train_cows))
print("Testing cows:", len(test_cows))

## 4. PCA: maximize overall variance

PCA does **not** use the infection labels when finding its components. We fit PCA on the standardized training features, inspect explained variance, and project the test observations using the same fitted transformation.

In [ ]:
pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

explained = pca.explained_variance_ratio_

variance_table = pd.DataFrame({
    "principal_component": [f"PC{i+1}" for i in range(len(explained))],
    "explained_variance_ratio": explained,
    "cumulative_variance": np.cumsum(explained)
})
display(variance_table.round(3))

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(range(1, len(explained)+1), explained, label="Individual")
plt.plot(range(1, len(explained)+1), np.cumsum(explained),
         marker="o", label="Cumulative")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("PCA explained variance")
plt.xticks(range(1, len(explained)+1))
plt.legend()
plt.show()

In [ ]:
pca_plot = pd.DataFrame({
    "PC1": X_train_pca[:, 0],
    "PC2": X_train_pca[:, 1],
    "cow_status": y_train.to_numpy()
})

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pca_plot, x="PC1", y="PC2", hue="cow_status", alpha=0.75)
plt.title("PCA projection of training milking events")
plt.show()

### Interpret PCA

- PC1 is the direction with the greatest variance in the training features.
- PC2 captures the greatest remaining variance while being orthogonal to PC1.
- The class colors are added **after** PCA. PCA did not use them to choose the directions.
- If the colors overlap, that does not mean PCA failed: its objective is variance, not class separation.

## 5. LDA: maximize separation between known classes

Unlike PCA, LDA uses the training labels. With two classes, LDA can produce at most one discriminant component (`number of classes - 1`).

We fit LDA using only the training data, then transform the test data.

In [ ]:
lda = LinearDiscriminantAnalysis(n_components=1)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)

print("LDA training projection shape:", X_train_lda.shape)
print("LDA test projection shape:", X_test_lda.shape)

In [ ]:
lda_plot = pd.DataFrame({
    "LD1": X_train_lda[:, 0],
    "cow_status": y_train.to_numpy()
})

plt.figure(figsize=(8, 3.5))
sns.histplot(data=lda_plot, x="LD1", hue="cow_status",
             element="step", stat="density", common_norm=False, bins=25)
plt.title("LDA: training observations projected onto LD1")
plt.show()

### Interpret LDA

LDA chooses a direction that seeks large separation between class means relative to variation within each class. The labels are used to learn this direction.

In this simplified example, the two distributions may separate strongly because we deliberately generated different patterns. Real data will be noisier, labels may be uncertain, and the result may be much less clean.

## 6. Use LDA as a classifier

LDA can also classify observations. We evaluate on held-out cows, not randomly held-out milking events. This is important because multiple events from the same cow are related.

In [ ]:
lda_classifier = LinearDiscriminantAnalysis()
lda_classifier.fit(X_train_scaled, y_train)

test_predictions = lda_classifier.predict(X_test_scaled)

print(classification_report(y_test, test_predictions))
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions,
    labels=["not_infected", "infected"],
    cmap="Blues"
)
plt.title("LDA classifier: held-out cows")
plt.show()

## 7. Compare PCA and LDA

| Question | PCA | LDA |
|---|---|---|
| Uses class labels to learn projection? | No | Yes |
| Main objective | Capture overall variance | Separate known classes |
| Components with two classes | Up to number of features | At most 1 |
| Can classify by itself? | No | Yes, in its classifier form |
